# TQ Sleeve Allocation Framework — α Engine

Milestone v2.0. PM-readable scorecard of options-overlay sleeve attractiveness across SPX/QQQ (and optionally XIU/XSP), backed by sleeve P&L backtests.

v1.0 legacy: `hmm.ipynb` (untouched).

## Section 0 — Setup & Config

In [ ]:
import pathlib
import warnings
import datetime as dt

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal

In [ ]:
UNDERLYINGS = {
    "SPX":  "SPX Index",
    "QQQ":  "QQQ US Equity",
    "XIU":  "XIU CN Equity",
    "XSP":  "XSP CN Equity",
}
REQUIRED_UNDERLYINGS = ("SPX", "QQQ")

IV_FIELDS = {
    "iv30_atm":   "30DAY_IMPVOL_100.0%MNY_DF",
    "iv30_90mny": "30DAY_IMPVOL_90.0%MNY_DF",
    "iv90_atm":   "90DAY_IMPVOL_100.0%MNY_DF",
}
PRICE_FIELD = "PX_LAST"

CROSS_ASSET = {
    "vix":     ("VIX Index",     "PX_LAST"),
    "vvix":    ("VVIX Index",    "PX_LAST"),
    "rf_rate": ("USGG3M Index",  "PX_LAST"),
}

START_DATE = "2005-01-01"
END_DATE   = dt.date.today().isoformat()

In [ ]:
CACHE_DIR = pathlib.Path.home() / "sleeve_alpha_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def nyse_index(start, end):
    sched = mcal.get_calendar("NYSE").schedule(start_date=start, end_date=end)
    return pd.DatetimeIndex(sched.index).normalize()

NYSE_INDEX = nyse_index(START_DATE, END_DATE)

## Section 1 — Data Layer

### 1.1 Cache wrapper

In [ ]:
def _cache_path(ticker, field, start, end):
    safe_t = ticker.replace(" ", "_").replace("/", "_")
    safe_f = field.replace("%", "pct").replace(".", "p").replace(" ", "_")
    return CACHE_DIR / f"{safe_t}__{safe_f}__{start}__{end}.parquet"

def _is_stale(path, max_age_trading_days=1):
    if not path.exists():
        return True
    mtime = dt.datetime.fromtimestamp(path.stat().st_mtime).date()
    today = dt.date.today()
    cal = mcal.get_calendar("NYSE")
    sched = cal.schedule(start_date=mtime, end_date=today)
    trading_days_since = max(0, len(sched) - 1)
    return trading_days_since > max_age_trading_days

def bdh_cached(ticker, field, start, end, force_refresh=False):
    path = _cache_path(ticker, field, start, end)
    if not force_refresh and not _is_stale(path):
        return pd.read_parquet(path)
    try:
        df = con.bdh(ticker, field, start_date=start, end_date=end)
    except Exception as e:
        if path.exists():
            warnings.warn(f"con.bdh failed for {ticker}/{field} ({e}); serving stale cache")
            return pd.read_parquet(path)
        raise
    if df is None or len(df) == 0:
        if path.exists():
            path.unlink()
        return pd.DataFrame()
    df.to_parquet(path)
    return df